In [ ]:
import json

import pandas as pd
from IPython.display import display

from study.data import record_cohort_step
from study.eicu import extract_events, numeric, priority
from study.io import DATA, RESOURCES, WORK, write_table
from study.readers import read_table

In [ ]:
RAW_FLOW_IDS = {
    'patients': 'uniquepid',
    'hospital_admissions': 'patienthealthsystemstayid',
    'icu_stays': 'patientunitstayid'
}
COHORT_FLOW_IDS = {'patients': 'subject_id', 'hospital_admissions': 'hadm_id', 'icu_stays': 'stay_id'}
ANCHOR = pd.Timestamp('2000-01-01')

In [ ]:
schema = json.loads((RESOURCES / 'mappings/eicu_schema.json').read_text())
mapping = pd.read_csv(RESOURCES / 'mappings/eicu_events.csv', keep_default_na = False)
icu_stays = read_table(DATA / 'eicu', 'patient', dtype = str, keep_default_na = False)

In [ ]:
# Select one hospital admission per person, then first ICU stay
cohort_flows = []
flow_ids = RAW_FLOW_IDS
icu_stays = icu_stays.copy()

# Standardise raw identifiers and admission offsets
for name in ('patientunitstayid', 'patienthealthsystemstayid', 'hospitalid'):
    icu_stays[name] = pd.to_numeric(icu_stays[name], errors = 'raise').astype('int64')

icu_stays['hospital_admission_offset'] = numeric(icu_stays.hospitaladmitoffset)

# Score hospital admissions and select one per patient
hospital_admissions = icu_stays[['uniquepid', 'patienthealthsystemstayid']].drop_duplicates()
hospital_admissions['selection_score'] = hospital_admissions.apply(
    lambda row: priority(f'{row.uniquepid}:{row.patienthealthsystemstayid}', 'index-admission-v1'), axis = 1
)
chosen_admissions = hospital_admissions.sort_values('selection_score').drop_duplicates('uniquepid')
candidate_icu_stays = record_cohort_step(
    cohort_flows,
    'eicu',
    'one_hospital_admission_per_patient',
    icu_stays,
    icu_stays.merge(chosen_admissions, on = ['uniquepid', 'patienthealthsystemstayid'], validate = 'many_to_one'),
    flow_ids,
)

# Select first ICU stay using the largest hospital admission offset
first_icu_stays = record_cohort_step(
    cohort_flows,
    'eicu',
    'first_icu_stay_in_selected_admission',
    candidate_icu_stays,
    candidate_icu_stays.sort_values(
        ['hospital_admission_offset', 'patientunitstayid'], ascending = [False, True]
    ).drop_duplicates('uniquepid'),
    flow_ids,
)

# Standardise demographics, outcomes, and stay times
stays = pd.DataFrame(
    {
        'domain': 'eicu',
        'subject_id': first_icu_stays.uniquepid,
        'hadm_id': first_icu_stays.patienthealthsystemstayid,
        'stay_id': first_icu_stays.patientunitstayid,
        'hospital_id': first_icu_stays.hospitalid.astype(str),
        'intime': ANCHOR,
        'outtime': ANCHOR + pd.to_timedelta(numeric(first_icu_stays.unitdischargeoffset), unit = 'm'),
        'los_days': numeric(first_icu_stays.unitdischargeoffset) / 1440,
        'hospital_minutes': numeric(first_icu_stays.hospitaldischargeoffset),
        'age': numeric(first_icu_stays.age.str.strip().replace({'> 89': '90', '>89': '90'})),
        'age_topcoded': first_icu_stays.age.str.contains('>', regex = False),
        'sex': first_icu_stays.gender.str.strip().str.upper().map({'FEMALE': 'F', 'MALE': 'M'}),
        'icu_type': first_icu_stays.unittype,
        'icu_type_code': first_icu_stays.unittype.map(schema['unit_mapping']),
        'in_hospital_mortality': first_icu_stays.hospitaldischargestatus.str.strip()
        .str.lower()
        .map({'alive': 0, 'expired': 1}),
        'height': numeric(first_icu_stays.admissionheight)
    }
)

# Retain adults with ICU and hospital stays of at least 48 hours and a recorded outcome
flow_ids = COHORT_FLOW_IDS
for step, keep in [
    ('age_at_least_18', stays.age.ge(18)),
    ('icu_stay_at_least_48_hours', stays.los_days.ge(2)),
    ('recorded_hospital_outcome', stays.in_hospital_mortality.isin([0, 1])),
    ('hospital_discharge_at_least_48_hours', stays.hospital_minutes.ge(2880))
]:
    stays = record_cohort_step(cohort_flows, 'eicu', step, stays, stays.loc[keep.loc[stays.index]], flow_ids)

# Assign calibration and test patients and extraction buckets
result = stays.drop(columns = 'hospital_minutes').reset_index(drop = True)
result['partition'] = result.subject_id.map(
    lambda value: 'calibration' if priority(value, 'external-calibration-v1') < 0.2 else 'test'
)
result['bucket'] = result.subject_id.map(lambda value: int(priority(value, 'bucket-v1') * 128))

result['in_hospital_mortality'] = result.in_hospital_mortality.astype(int)
result['icu_type_code'] = pd.to_numeric(result.icu_type_code, errors = 'raise').astype(float)
stays = result

In [ ]:
# Save selected cohort and selection counts
root = WORK / 'data/eicu'
write_table(root / 'cohort.parquet', stays)

flow = pd.DataFrame(cohort_flows)
flow.to_csv(root / 'cohort_flow.csv', index = False)

display(flow)
display(stays.groupby('partition').size())

In [ ]:
# Extract accepted measurements in patient buckets
extract_events(DATA / 'eicu', schema['events'], stays, mapping, root / 'events')